# Student-Hybrid: MobileNetV3 Backbone + Shallow Cross-Attention

This notebook implements **Student-Hybrid**, a student network that keeps LoFTR's attention-based feature interaction but replaces the heavy ResNet-18 FPN backbone with a lightweight MobileNetV3-Small trunk.

## Architecture Overview

| Component | Teacher (LoFTR) | Student-CNN | Student-Hybrid |
|---|---|---|---|
| Backbone | ResNet-18 + FPN (~11M) | MobileNetV3-Small (~0.014M) | MobileNetV3-Small (~0.014M) |
| Coarse features | 256-dim, 1/8 res | 128-dim, 1/8 res | 128-dim, 1/8 res |
| Fine features | 128-dim, 1/2 res | 32-dim, 1/2 res | 32-dim, 1/2 res |
| Feature interaction | 4 rounds, 8 heads | Dilated convs (rates 1,2,4,8) | 2 rounds, 4 heads |
| Coarse matching | Dual-softmax | Dual-softmax | Dual-softmax |
| Fine refinement | Local window correlation | Local window correlation | Local window correlation |

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np

from models import StudentHybrid, StudentHybridConfig
from losses import DistillationLoss

## 1. Model Configuration

All hyperparameters are centralized in `StudentHybridConfig`. Key settings:
- Backbone: MobileNetV3-Small, features[0:4], coarse at 1/8 resolution
- Attention: 2 rounds of self+cross attention, 4 heads, d_model=128 (half of LoFTR: 4 rounds / 8 heads)
- Dual-softmax temperature: 0.1
- Fine refinement window: 5x5

In [2]:
config = StudentHybridConfig()

print('=== Student-Hybrid Configuration ===')
print(f'Coarse feature dim:    {config.coarse_dim} (teacher: {config.teacher_coarse_dim})')
print(f'Fine feature dim:      {config.fine_dim} (teacher: {config.teacher_fine_dim})')
print(f'Attention heads:       {config.nhead} (LoFTR uses 8)')
print(f'Attention rounds:      {config.n_rounds} (LoFTR uses 4)')
print(f'Temperature:           {config.temperature}')
print(f'Match threshold:       {config.match_threshold}')
print(f'Fine window size:      {config.fine_window_size}')
print(f'\nDistillation weights:')
print(f'  lambda_coarse_kd = {config.lambda_coarse_kd}')
print(f'  lambda_feat_kd   = {config.lambda_feat_kd}')
print(f'  lambda_fine_kd   = {config.lambda_fine_kd}')
print(f'  lambda_gt        = {config.lambda_gt}')

=== Student-Hybrid Configuration ===
Coarse feature dim:    128 (teacher: 256)
Fine feature dim:      32 (teacher: 128)
Attention heads:       4 (LoFTR uses 8)
Attention rounds:      2 (LoFTR uses 4)
Temperature:           0.1
Match threshold:       0.2
Fine window size:      5

Distillation weights:
  lambda_coarse_kd = 1.0
  lambda_feat_kd   = 0.5
  lambda_fine_kd   = 0.5
  lambda_gt        = 1.0


## 2. Instantiate Model

The Student-Hybrid model consists of:
1. **MobileNetV3Backbone**: truncated MobileNetV3-Small (features[0:4]), ~0.014M params
2. **ShallowCrossAttention**: 2 rounds of self-attention + cross-attention with 4 heads
3. **CoarseMatching**: dual-softmax confidence matrix + mutual NN extraction
4. **FineRefinement**: local window correlation with spatial expectation
5. **feat_projector**: 1x1 conv for distillation (128 -> 256 channels, training only)

In [9]:
model = StudentHybrid(config)

components = {
    'backbone':        model.backbone,
    'attention':       model.attention,
    'coarse_matching': model.coarse_matching,
    'fine_refinement': model.fine_refinement,
    'feat_projector':  model.feat_projector,
}

print('=== Parameter Count ===')
total = 0
for name, module in components.items():
    n = sum(p.numel() for p in module.parameters())
    total += n
    print(f'  {name:20s}: {n:>10,d} ({n/1e6:.3f}M)')
print(f'  {"total":20s}: {total:>10,d} ({total/1e6:.3f}M)')

proj_params = sum(p.numel() for p in model.feat_projector.parameters())
inference_params = total - proj_params
print(f'\nNote: feat_projector is only used during training (discarded at inference)')
print(f'Inference parameters: {inference_params:,d} ({inference_params/1e6:.3f}M)')

=== Parameter Count ===
  backbone            :     13,944 (0.014M)
  attention           :    397,568 (0.398M)
  coarse_matching     :          0 (0.000M)
  fine_refinement     :      1,056 (0.001M)
  feat_projector      :     32,768 (0.033M)
  total               :    445,336 (0.445M)

Note: feat_projector is only used during training (discarded at inference)
Inference parameters: 412,568 (0.413M)


## 3. Forward Pass Test (Training Mode)

In training mode, the model outputs:
- `conf_matrix`: (B, N0, N1) coarse confidence matrix
- `coarse_feat0/1`: (B, 128, H/8, W/8) pre-attention coarse features for distillation
- `coarse_feat0/1_proj`: (B, 256, H/8, W/8) projected features for feature-level KD

In [10]:
# Batch of 2 image pairs, 480x480
B, H, W = 2, 480, 480
data_train = {
    'image0': torch.randn(B, 1, H, W),
    'image1': torch.randn(B, 1, H, W),
}

model.train()
with torch.no_grad():
    out_train = model(data_train)

print('=== Training Mode Outputs ===')
for key in ['conf_matrix', 'coarse_feat0', 'coarse_feat1',
            'coarse_feat0_proj', 'coarse_feat1_proj']:
    if key in out_train:
        v = out_train[key]
        print(f'  {key:30s}: {str(v.shape):25s} dtype={v.dtype}')

cm = out_train['conf_matrix']
Hc = H // 8
print(f'\nCoarse grid size: {Hc}x{Hc} = {Hc*Hc} locations per image')
print(f'Confidence matrix shape: {cm.shape}  (expected ({B}, {Hc*Hc}, {Hc*Hc}))')
print(f'Confidence range: [{cm.min():.6f}, {cm.max():.6f}]')

=== Training Mode Outputs ===
  conf_matrix                   : torch.Size([2, 3600, 3600]) dtype=torch.float32
  coarse_feat0                  : torch.Size([2, 128, 60, 60]) dtype=torch.float32
  coarse_feat1                  : torch.Size([2, 128, 60, 60]) dtype=torch.float32
  coarse_feat0_proj             : torch.Size([2, 256, 60, 60]) dtype=torch.float32
  coarse_feat1_proj             : torch.Size([2, 256, 60, 60]) dtype=torch.float32

Coarse grid size: 60x60 = 3600 locations per image
Confidence matrix shape: torch.Size([2, 3600, 3600])  (expected (2, 3600, 3600))
Confidence range: [0.000000, 0.011722]


## 4. Forward Pass Test (Inference Mode)

In inference mode the model additionally outputs:
- `keypoints0/1`: (M, 2) matched keypoint coordinates at original resolution
- `confidence`: (M,) match confidence scores

Output interface is compatible with the LoFTR teacher.

In [11]:
data_eval = {
    'image0': torch.randn(1, 1, H, W),
    'image1': torch.randn(1, 1, H, W),
}

model.eval()
with torch.no_grad():
    out_eval = model(data_eval)

print('=== Inference Mode Outputs ===')
print(f'  keypoints0 shape: {out_eval["keypoints0"].shape}')
print(f'  keypoints1 shape: {out_eval["keypoints1"].shape}')
print(f'  confidence shape: {out_eval["confidence"].shape}')
n_matches = len(out_eval['keypoints0'])
print(f'\nNumber of matches found: {n_matches}')

if n_matches > 0:
    print(f'\nFirst 5 matches (keypoints0 -> keypoints1):')
    for i in range(min(5, n_matches)):
        kp0 = out_eval['keypoints0'][i].numpy()
        kp1 = out_eval['keypoints1'][i].numpy()
        conf = out_eval['confidence'][i].item()
        print(f'  ({kp0[0]:.1f}, {kp0[1]:.1f}) -> ({kp1[0]:.1f}, {kp1[1]:.1f})  conf={conf:.4f}')
else:
    print('(No matches on random input -- expected behaviour)')

=== Inference Mode Outputs ===
  keypoints0 shape: torch.Size([0, 2])
  keypoints1 shape: torch.Size([0, 2])
  confidence shape: torch.Size([0])

Number of matches found: 0
(No matches on random input -- expected behaviour)


## 5. Distillation Loss

Multi-level distillation loss with 4 components:

| Loss | Target | Method | Weight |
|---|---|---|---|
| L_coarse_kd | Teacher's dual-softmax confidence | KL divergence | lambda_1=1.0 |
| L_feat_kd | Intermediate feature representations | MSE + 1x1 conv projector | lambda_2=0.5 |
| L_fine_kd | Sub-pixel refined coordinates | L1 distance | lambda_3=0.5 |
| L_gt | GT correspondences (homography) | Focal loss + L2 | lambda_4=1.0 |

In [12]:
loss_fn = DistillationLoss(config)

# Student training forward
model.train()
data_train = {
    'image0': torch.randn(B, 1, H, W),
    'image1': torch.randn(B, 1, H, W),
}
with torch.no_grad():
    student_out = model(data_train)

# Mock teacher outputs (in practice these come from frozen LoFTR)
Hc = H // 8
teacher_out = {
    'conf_matrix':          torch.rand(B, Hc * Hc, Hc * Hc),
    'teacher_coarse_feat0': torch.randn(B, config.teacher_coarse_dim, Hc, Hc),
    'teacher_coarse_feat1': torch.randn(B, config.teacher_coarse_dim, Hc, Hc),
}

losses = loss_fn(student_out, teacher_out)

print('=== Distillation Loss Components ===')
for name, val in losses.items():
    print(f'  {name:15s}: {val.item():.6f}')

=== Distillation Loss Components ===
  coarse_kd      : 4.282454
  feat_kd        : 1.394101
  fine_kd        : 0.000000
  gt_sup         : 0.000000
  total          : 4.979505


## 6. Model Architecture Summary

In [13]:
print(model)

StudentHybrid(
  (backbone): MobileNetV3Backbone(
    (fine_layers): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
    )
    (coarse_layers): Sequential(
      (0): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (activation): ReLU()
            (scale_activ

## 7. Comparison with LoFTR Teacher and Student-CNN

The key experimental question: **Does keeping shallow attention (while drastically shrinking the backbone) preserve matching quality better than replacing attention with dilated convolutions?**

### What Student-Hybrid preserves from LoFTR:
- Self-attention + cross-attention feature interaction
- Coarse-to-fine matching pipeline
- Dual-softmax confidence matrix
- Local window fine refinement

### What Student-Hybrid replaces:
- ResNet-18 FPN backbone -> MobileNetV3-Small (features[0:4], ~14K params)
- 4 attention rounds, 8 heads, d_model=256 -> 2 rounds, 4 heads, d_model=128
- 256-dim coarse features -> 128-dim (with 1x1 projector for distillation)

### Student-Hybrid vs Student-CNN:
- Both share the **same MobileNetV3Backbone** (identical class, identical init)
- Student-Hybrid keeps attention -> better on low-texture / large viewpoint change
- Student-CNN uses dilated convs -> faster inference (no quadratic attention cost)

### Expected trade-offs:
- Total params ~0.445M vs LoFTR ~12M (~27x smaller)
- Retains global context via attention (unlike Student-CNN)
- Attention is O(N^2) in sequence length -- slower at inference than Student-CNN
- Accuracy gap vs LoFTR mainly from reduced backbone capacity, not attention depth

In [14]:
# FLOPs estimation (approximate)
try:
    from thop import profile
    model.eval()
    dummy_data = {'image0': torch.randn(1, 1, 480, 480), 'image1': torch.randn(1, 1, 480, 480)}
    flops, params = profile(model, inputs=(dummy_data,), verbose=False)
    print(f'FLOPs:  {flops/1e9:.2f} GFLOPs')
    print(f'Params: {params/1e6:.3f} M')
except ImportError:
    print("Install 'thop' for FLOPs estimation: pip install thop")
    total = sum(p.numel() for p in model.parameters())
    proj  = sum(p.numel() for p in model.feat_projector.parameters())
    print(f'Total params:     {total:,} ({total/1e6:.3f}M)')
    print(f'Inference params: {total - proj:,} ({(total - proj)/1e6:.3f}M)')

Install 'thop' for FLOPs estimation: pip install thop
Total params:     445,336 (0.445M)
Inference params: 412,568 (0.413M)
